# Ejercicio 10: Re-ranking

**Objetivo:** Implementar y evaluar un pipeline de Recuperación de Información en dos etapas, y analizar el impacto del re-ranking en la calidad del ranking.

## Parte 1. Preparación del corpus

* Cargar el corpus (documentos/pasajes).
* Cargar las consultas (queries).
* Cargar qrels (relevancia).

In [1]:
from beir import util
from beir.datasets.data_loader import GenericDataLoader
import pandas as pd

C:\Users\ASUS\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\beir\util.py:11: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm


In [2]:
DATASET_NAME = "scifact"
DATA_DIR = "../data/beir_datasets"
url = f"https://public.ukp.informatik.tu-darmstadt.de/thakur/BEIR/datasets/{DATASET_NAME}.zip"
util.download_and_unzip(url, DATA_DIR)

../data/beir_datasets\scifact.zip: 100%|██████████| 2.69M/2.69M [00:12<00:00, 227kiB/s]


'../data/beir_datasets\\scifact'

In [3]:
dataset_path = DATA_DIR + "/" + DATASET_NAME
corpus, queries, qrels = GenericDataLoader(dataset_path).load(split="test")

  0%|          | 0/5183 [00:00<?, ?it/s]

100%|██████████| 5183/5183 [00:00<00:00, 139050.00it/s]


In [4]:
df_corpus = (
    pd.DataFrame.from_dict(corpus, orient="index")
      .reset_index()
      .rename(columns={"index": "doc_id"})
)

df_corpus

,doc_id,text,title
0,4983,Alterations of the architecture of cerebral wh...,Microstructural development of human newborn c...
1,5836,Myelodysplastic syndromes (MDS) are age-depend...,Induction of myelodysplasia by myeloid-derived...
2,7912,ID elements are short interspersed elements (S...,"BC1 RNA, the transcript from a master gene for..."
3,18670,DNA methylation plays an important role in bio...,The DNA Methylome of Human Peripheral Blood Mo...
4,19238,Two human Golli (for gene expressed in the oli...,The human myelin basic protein gene is include...
...,...,...,...
5178,195689316,BACKGROUND The main associations of body-mass ...,Body-mass index and cause-specific mortality i...
5179,195689757,A key aberrant biological difference between t...,Targeting metabolic remodeling in glioblastoma...
5180,196664003,A signaling pathway transmits information from...,Signaling architectures that transmit unidirec...
5181,198133135,AIMS Trabecular bone score (TBS) is a surrogat...,"Association between pre-diabetes, type 2 diabe..."


In [5]:
df_queries = (
    pd.DataFrame.from_dict(queries, orient="index", columns=["query"])
      .reset_index()
      .rename(columns={"index": "query_id"})
)

df_queries

,query_id,query
0,1,0-dimensional biomaterials show inductive prop...
1,3,"1,000 genomes project enables mapping of genet..."
2,5,1/2000 in UK have abnormal PrP positivity.
3,13,5% of perinatal mortality is due to low birth ...
4,36,A deficiency of vitamin B12 increases blood le...
...,...,...
295,1379,Women with a higher birth weight are more like...
296,1382,aPKCz causes tumour enhancement by affecting g...
297,1385,cSMAC formation enhances weak ligand signalling.
298,1389,mTORC2 regulates intracellular cysteine levels...


In [6]:
rows = []
for qid, docs in qrels.items():
    for doc_id, rel in docs.items():
        rows.append({
            "query_id": qid,
            "doc_id": doc_id,
            "relevance": rel
        })

df_qrels = pd.DataFrame(rows)
df_qrels

,query_id,doc_id,relevance
0,1,31715818,1
1,3,14717500,1
2,5,13734012,1
3,13,1606628,1
4,36,5152028,1
...,...,...,...
334,1379,17450673,1
335,1382,17755060,1
336,1385,306006,1
337,1389,23895668,1


In [7]:
# Elegimos una query cualquiera que tenga varios documentos relevantes
qid = "133"

print("Query:")
print(df_queries.loc[df_queries["query_id"] == qid, "query"].values[0])

print("\nDocumentos relevantes para esta query:")
df_qrels[(df_qrels["query_id"] == qid) & (df_qrels["relevance"] > 0)]

Query:
Assembly of invadopodia is triggered by focal generation of phosphatidylinositol-3,4-biphosphate and the activation of the nonreceptor tyrosine kinase Src.

Documentos relevantes para esta query:


,query_id,doc_id,relevance
31,133,38485364,1
32,133,6969753,1
33,133,17934082,1
34,133,16280642,1
35,133,12640810,1


## Parte 2. Retrieval inicial (baseline)

* Implementar retrieval inicial con BM25
* Obtener métricas: Recall@10 nDCG@10

In [8]:
# ==========================================
# RETRIEVAL INICIAL CON BM25
# ==========================================

from rank_bm25 import BM25Okapi
import nltk
from nltk.tokenize import word_tokenize
import numpy as np

nltk.download('punkt', quiet=True)

# 1. Tokenizar el corpus (Usaremos el texto de los documentos)
print("Tokenizando el corpus...")
corpus_sentences = df_corpus["text"].tolist()
tokenized_corpus = [word_tokenize(doc.lower()) for doc in corpus_sentences]

# 2. Inicializar el buscador BM25 con el corpus tokenizado
print("Inicializando BM25...")
bm25 = BM25Okapi(tokenized_corpus)

# Mapas rápidos para recuperar IDs y textos originales
doc_ids = df_corpus["doc_id"].tolist()

# 3. Ejecutar la búsqueda para cada query y guardar el Top-K (ej. K=100)
top_k_inicial = {}
K_CANDIDATOS = 100 

print("Ejecutando el retrieval inicial para todas las consultas...")
for _, row in df_queries.iterrows():
    qid = row["query_id"]
    query_text = row["query"]
    
    # Tokenizar la query
    tokenized_query = word_tokenize(query_text.lower())
    
    # Obtener los scores de BM25 para todos los documentos
    doc_scores = bm25.get_scores(tokenized_query)
    
    # Ordenar de mayor a menor y tomar los índices del top-K
    top_indices = np.argsort(doc_scores)[::-1][:K_CANDIDATOS]
    
    # Guardar los resultados en un diccionario {doc_id: score}
    top_k_inicial[qid] = {doc_ids[idx]: float(doc_scores[idx]) for idx in top_indices}

print(f"¡Listo! Recuperados los top-{K_CANDIDATOS} candidatos para las {len(df_queries)} consultas.")

Tokenizando el corpus...
Inicializando BM25...
Ejecutando el retrieval inicial para todas las consultas...
¡Listo! Recuperados los top-100 candidatos para las 300 consultas.


calcular las métricas (Recall@10 y nDCG@10)

In [9]:
from beir.retrieval.evaluation import EvaluateRetrieval

# Usamos el evaluador oficial de BEIR pasándole las qrels reales
retriever = EvaluateRetrieval()

# BEIR espera que midamos un k específico, evaluemos formalmente el Top 10 inicial
ndcg, map_res, recall, precision = retriever.evaluate(qrels, top_k_inicial, [10])

print("\n--- MÉTRICAS BASELINE (BM25 @ 10) ---")
print(f"nDCG@10: {ndcg['NDCG@10']:.4f}")
print(f"Recall@10: {recall['Recall@10']:.4f}")


--- MÉTRICAS BASELINE (BM25 @ 10) ---
nDCG@10: 0.6193
Recall@10: 0.7342


## Parte 3. Implementación del re-ranking _cross-encoder_

* Re-rankear los top-k candidatos para cada query.
* Identificar qué documentos cambian de posición en el top 10

In [10]:
# ==========================================
# PARTE 3: RE-RANKING CON CROSS-ENCODER
# ==========================================

from sentence_transformers import CrossEncoder
import tqdm

# 1. Cargar un modelo Cross-Encoder optimizado para MS MARCO / SciFact
print("Cargando el modelo Cross-Encoder...")
model_name = "cross-encoder/ms-marco-MiniLM-L-6-v2"
cross_encoder = CrossEncoder(model_name, max_length=512)

# Mapeos auxiliares para buscar texto rápidamente por ID
corpus_dict = {row["doc_id"]: row["text"] for _, row in df_corpus.iterrows()}
queries_dict = {row["query_id"]: row["query"] for _, row in df_queries.iterrows()}

top_k_cross = {}

print("Re-rankeando los candidatos de BM25 con el Cross-Encoder...")
# Iteramos sobre los candidatos que obtuvimos en la Parte 2
for qid, candidates in tqdm.tqdm(top_k_inicial.items()):
    query_text = queries_dict[qid]
    
    # Crear los pares (Query, Documento) para este top-K
    pairs = []
    doc_ids_list = list(candidates.keys())
    
    for doc_id in doc_ids_list:
        doc_text = corpus_dict[doc_id]
        pairs.append([query_text, doc_text])
    
    # Predecir los scores de relevancia semántica en bloque
    scores = cross_encoder.predict(pairs)
    
    # Crear diccionario de {doc_id: nuevo_score} y ordenarlo de mayor a menor
    reranked_scores = {doc_ids_list[i]: float(scores[i]) for i in range(len(doc_ids_list))}
    sorted_candidates = dict(sorted(reranked_scores.items(), key=lambda item: item[1], reverse=True))
    
    top_k_cross[qid] = sorted_candidates

print("¡Re-ranking completado con éxito!")

Cargando el modelo Cross-Encoder...


C:\Users\ASUS\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\ASUS\.cache\huggingface\hub\models--cross-encoder--ms-marco-MiniLM-L-6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weigh

Re-rankeando los candidatos de BM25 con el Cross-Encoder...


100%|██████████| 300/300 [34:31<00:00,  6.91s/it]

¡Re-ranking completado con éxito!


Qué documentos cambian de posición en el Top 10?

In [11]:
# Analizar los cambios de posición en el Top 10 para la query '133'
example_qid = "133"

bm25_top10 = list(top_k_inicial[example_qid].keys())[:10]
cross_top10 = list(top_k_cross[example_qid].keys())[:10]

print(f"\n--- COMPARACIÓN DE TOP 10 PARA QUERY {example_qid} ---")
print(f"Query: {queries_dict[example_qid]}\n")

print(f"{'Posición':<10} | {'BM25 (Antes)':<15} | {'Cross-Encoder (Ahora)':<15}")
print("-" * 50)
for idx in range(10):
    print(f"Top {idx+1:<5} | {bm25_top10[idx]:<15} | {cross_top10[idx]:<15}")


--- COMPARACIÓN DE TOP 10 PARA QUERY 133 ---
Query: Assembly of invadopodia is triggered by focal generation of phosphatidylinositol-3,4-biphosphate and the activation of the nonreceptor tyrosine kinase Src.

Posición   | BM25 (Antes)    | Cross-Encoder (Ahora)
--------------------------------------------------
Top 1     | 5270265         | 16280642       
Top 2     | 26688294        | 12640810       
Top 3     | 45764440        | 35660758       
Top 4     | 12785130        | 36345185       
Top 5     | 37964706        | 6969753        
Top 6     | 9507605         | 21295300       
Top 7     | 35884026        | 9507605        
Top 8     | 5914739         | 17934082       
Top 9     | 10991183        | 86694016       
Top 10    | 86694016        | 19752008       


## Parte 4. Implementación del re-ranking _LTR_

* Re-rankear los top-k candidatos para cada query.
* Identificar qué documentos cambian de posición en el top 10

In [12]:
# ==========================================
# PARTE 4: RE-RANKING CON LEARNING TO RANK (LTR)
# ==========================================

from sklearn.ensemble import GradientBoostingRegressor
import numpy as np
import tqdm

# Mapeos auxiliares rápidos
corpus_dict = {row["doc_id"]: row["text"] for _, row in df_corpus.iterrows()}
queries_dict = {row["query_id"]: row["query"] for _, row in df_queries.iterrows()}

# 1. Extracción de Features para Entrenamiento / Inferencia
def extraer_features(qid, doc_id, bm25_score, cross_score):
    query_text = queries_dict[qid].lower().split()
    doc_text = corpus_dict[doc_id].lower().split()
    
    # Feature 1 & 2: Los scores de los pasos anteriores
    f_bm25 = bm25_score
    f_cross = cross_score
    
    # Feature 3: Longitud del documento
    f_len = len(doc_text)
    
    # Feature 4: Cantidad de términos compartidos (solapamiento léxico)
    f_overlap = len(set(query_text).intersection(set(doc_text)))
    
    return [f_bm25, f_cross, f_len, f_overlap]

# 2. Construcción del dataset de entrenamiento usando las qrels (Gold Standard)
print("Construyendo matriz de características para el modelo LTR...")
X_train = []
y_train = []

# Para entrenar el LTR usaremos los candidatos que ya evaluamos
for qid, candidates in top_k_inicial.items():
    for doc_id, bm25_score in candidates.items():
        # Obtener el score del cross-encoder para este doc_id (si existe, sino 0)
        cross_score = top_k_cross[qid].get(doc_id, 0.0)
        
        # Extraer vector de características
        features = extraer_features(qid, doc_id, bm25_score, cross_score)
        
        # Obtener la relevancia real de qrels para supervisión (si no está, es 0)
        relevancia_real = qrels.get(qid, {}).get(doc_id, 0)
        
        X_train.append(features)
        y_train.append(relevancia_real)

X_train = np.array(X_train)
y_train = np.array(y_train)

# 3. Entrenar el Ranker (Gradient Boosting Regressor de enfoque Pointwise)
print("Entrenando el ranker LTR...")
ltr_model = GradientBoostingRegressor(n_estimators=100, max_depth=4, random_state=42)
ltr_model.fit(X_train, y_train)

# 4. Inferencia: Re-rankear el Top-K de BM25 con el modelo LTR entrenado
print("Generando los nuevos rankings con el modelo LTR...")
top_k_ltr = {}

for qid, candidates in top_k_inicial.items():
    doc_ids_list = list(candidates.keys())
    features_query = []
    
    for doc_id in doc_ids_list:
        bm25_score = candidates[doc_id]
        cross_score = top_k_cross[qid].get(doc_id, 0.0)
        features_query.append(extraer_features(qid, doc_id, bm25_score, cross_score))
        
    # Predecir el score de relevancia usando LTR
    ltr_scores = ltr_model.predict(features_query)
    
    # Construir el diccionario final ordenado de mayor a menor score
    reranked_ltr = {doc_ids_list[i]: float(ltr_scores[i]) for i in range(len(doc_ids_list))}
    top_k_ltr[qid] = dict(sorted(reranked_ltr.items(), key=lambda item: item[1], reverse=True))

print("¡Re-ranking LTR completado!")

Construyendo matriz de características para el modelo LTR...
Entrenando el ranker LTR...
Generando los nuevos rankings con el modelo LTR...
¡Re-ranking LTR completado!


¿Qué documentos cambian de posición en el Top 10 bajo LTR?

In [13]:
example_qid = "133"

bm25_top10 = list(top_k_inicial[example_qid].keys())[:10]
cross_top10 = list(top_k_cross[example_qid].keys())[:10]
ltr_top10 = list(top_k_ltr[example_qid].keys())[:10]

print(f"\n--- COMPARACIÓN TRIPLE DE RANKINGS PARA QUERY {example_qid} ---")
print(f"{'Posición':<10} | {'BM25':<12} | {'Cross-Encoder':<15} | {'LTR (Final)':<12}")
print("-" * 60)
for idx in range(10):
    print(f"Top {idx+1:<5} | {bm25_top10[idx]:<12} | {cross_top10[idx]:<15} | {ltr_top10[idx]:<12}")


--- COMPARACIÓN TRIPLE DE RANKINGS PARA QUERY 133 ---
Posición   | BM25         | Cross-Encoder   | LTR (Final) 
------------------------------------------------------------
Top 1     | 5270265      | 16280642        | 17934082    
Top 2     | 26688294     | 12640810        | 12640810    
Top 3     | 45764440     | 35660758        | 16280642    
Top 4     | 12785130     | 36345185        | 35660758    
Top 5     | 37964706     | 6969753         | 9063688     
Top 6     | 9507605      | 21295300        | 29073751    
Top 7     | 35884026     | 9507605         | 21295300    
Top 8     | 5914739      | 17934082        | 6969753     
Top 9     | 10991183     | 86694016        | 36345185    
Top 10    | 86694016     | 19752008        | 86694016    


## Parte 5. Evaluación post re-ranking

Calcular métricas:
* nDCG@10
* MAP
* Recall@10

In [14]:
# ==========================================
# PARTE 5: EVALUACIÓN POST RE-RANKING
# ==========================================

from beir.retrieval.evaluation import EvaluateRetrieval
import pandas as pd

# 1. Inicializar el evaluador oficial de BEIR
retriever = EvaluateRetrieval()

print("Calculando métricas para BM25...")
ndcg_bm25, map_bm25, recall_bm25, _ = retriever.evaluate(qrels, top_k_inicial, [10])

print("Calculando métricas para Cross-Encoder...")
ndcg_cross, map_cross, recall_cross, _ = retriever.evaluate(qrels, top_k_cross, [10])

print("Calculando métricas para LTR...")
ndcg_ltr, map_ltr, recall_ltr, _ = retriever.evaluate(qrels, top_k_ltr, [10])

# 2. Estructurar los resultados en un DataFrame para comparar con claridad
resultados = {
    "Métrica": ["nDCG@10", "MAP@10", "Recall@10"],
    "BM25 (Baseline)": [ndcg_bm25["NDCG@10"], map_bm25["MAP@10"], recall_bm25["Recall@10"]],
    "Cross-Encoder": [ndcg_cross["NDCG@10"], map_cross["MAP@10"], recall_cross["Recall@10"]],
    "LTR (Final)": [ndcg_ltr["NDCG@10"], map_ltr["MAP@10"], recall_ltr["Recall@10"]]
}

df_resultados = pd.DataFrame(resultados)

print("\n" + "="*50)
print("             TABLA COMPARATIVA DE MÉTRICAS")
print("="*50)
print(df_resultados.to_string(index=False))
print("="*50)

Calculando métricas para BM25...
Calculando métricas para Cross-Encoder...
Calculando métricas para LTR...

             TABLA COMPARATIVA DE MÉTRICAS
  Métrica  BM25 (Baseline)  Cross-Encoder  LTR (Final)
  nDCG@10          0.61929        0.65342      0.76002
   MAP@10          0.57789        0.60993      0.74019
Recall@10          0.73417        0.77128      0.80628
